In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import base64
import os
import json
import os
from io import BytesIO
from PIL import Image

In [2]:
# Install required libraries for Qwen3-VL-2B-Instruct
!pip install -q torchvision accelerate transformers>=4.45.0

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

QWEN_MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"

# Initialize processor and model
processor_qwen = AutoProcessor.from_pretrained(QWEN_MODEL_NAME)

model_qwen = AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to("cuda")

model_qwen.eval()

print("Qwen3-VL-2B-Instruct loaded successfully!")

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Qwen3-VL-2B-Instruct loaded successfully!


In [3]:
def query_from_path(image_path):
    # Helper to look up the correct query based on the image_path file name
    target_name = "/images/" + os.path.basename(image_path)
    for record in annotations_list:
        if record["image_name"] == target_name:
            return record["query"]
    raise ValueError(f"No annotation record found for image path: {image_path}")

In [4]:
def get_base64_from_rgb(rgb_image):
    # Helper to convert in-memory numpy array (RGB) to base64 string
    bgr_img = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
    _, buffer = cv2.imencode('.jpg', bgr_img)
    return base64.b64encode(buffer).decode('utf-8')

In [5]:
def evaluate_original(image_path):
  with open(image_path, "rb") as f:
    base64_image = base64.b64encode(f.read()).decode("utf-8")
  query_text = query_from_path(image_path)
  # Predict
  result = evaluate(query_text,base64_image)
  return result

In [6]:

def evaluate(query_text, b64_str):
    # Decode the base64 string to a PIL Image
    image_bytes = base64.b64decode(b64_str)
    image = Image.open(BytesIO(image_bytes)).convert("RGB")

    question = query_text + "\nAnswer only Yes or No."

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question}
            ]
        }
    ]

    text = processor_qwen.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor_qwen(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    ).to(model_qwen.device)

    with torch.inference_mode():
        output_ids = model_qwen.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    generated_ids = output_ids[:, len(inputs["input_ids"][0]):]

    answer = processor_qwen.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return answer.strip()

In [7]:
import os
import pandas as pd
from tqdm.notebook import tqdm

# --- Paths Configuration (Adjust as needed) ---
IMAGE_DIR = "/kaggle/input/datasets/ramakanthjogada/pope-dataset/images/images"
ANNOTATIONS_PATH = "/kaggle/input/datasets/ramakanthjogada/pope-dataset/annotations.json"

# Load annotations lookup map for faster access
with open(ANNOTATIONS_PATH, "r") as f:
    annotations_list = json.load(f)

# Create a lookup map for ground truth: (image_id) -> label
# Since we are evaluating the actual files in /images/, we target '/images/' records
ground_truth_map = {}
for record in annotations_list:
    if record["image_name"].startswith("/images/"):
        img_id = record["image_id"]
        ground_truth_map[img_id] = record["label"]

# Get list of all images in the directory
image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]

results = []

print(f"Processing {len(image_files)} images through all variations...")
for filename in tqdm(image_files):
    # Extract image_id from filename (e.g., '000000046207.jpg' -> 46207)
    try:
        image_id = int(os.path.splitext(filename)[0])
    except ValueError:
        continue

    image_path = os.path.join(IMAGE_DIR, filename)
    ground_truth = ground_truth_map.get(image_id, "Unknown")

    # Evaluate under all conditions
    try:
        orig_val = evaluate_original(image_path)
    except Exception as e:
        orig_val = f"Error: {str(e)}"
         
        
        
    results.append({
        "Image_Id": image_id,
        "Original": orig_val})

df_results = pd.DataFrame(results)
display(df_results.head(10))

Processing 500 images through all variations...


  0%|          | 0/500 [00:00<?, ?it/s]

,Image_Id,Original
0,169361,No
1,47022,No
2,556064,No
3,474004,No
4,306710,No
5,33882,No
6,195213,Yes
7,196545,No
8,186703,No
9,215863,No


In [8]:
import pandas as pd
df = pd.read_csv("/kaggle/input/datasets/ramakanthjogada/diff-model-evals/diff_models_eval.csv")

merged_df = pd.merge(df, df_results, on="Image_Id", how="outer")
merged_df.head()

,Image_Id,Ground_truth,Qwen-3B,Smol-2.2B,Original
0,6578,No,Yes.,Yes.,No
1,8602,No,No,No.,No
2,9469,No,No,No.,No
3,9712,No,Yes.,No.,Yes
4,10733,No,No,No.,No


In [9]:
merged_df.rename(columns={
    "Original": "Qwen-2B",
}, inplace=True)

merged_df.to_csv("diff_models_eval.csv", index=False)